# MQTT Subscriber + Upload on Redis

In [1]:
# Import the required modules
import paho.mqtt.client as mqtt
import redis
import json

## Establish a connection with Redis database

In [2]:
# Redis Database parameters
REDIS_HOST = 'redis-10264.c326.us-east-1-3.ec2.redns.redis-cloud.com'
REDIS_PORT = '10264'
REDIS_USERNAME = 'default'
REDIS_PASSWORD = '6Klkkhocow62bA84ZCn0Kj1tLJUJU0EW'

# Establish a connection to the database and check if the connection works
redis_client = redis.Redis(host = REDIS_HOST, 
                           port = REDIS_PORT, 
                           username = REDIS_USERNAME, 
                           password = REDIS_PASSWORD) 

is_connected = redis_client.ping()
print('Connect:', is_connected)

Connect: True


## Define the callback for when the client receives a response from the MQTT broker 

In [3]:
def on_connect(client, userdata, flags, rc):
    print(f'Connected with result code {str(rc)}')
    # Subscribe to a topic when the client connects
    client.subscribe('s345139')

## Define the callback for when a message is published on a subscribed topic

In [4]:
def on_message(client, userdata, msg):
    # Decode the message
    message = msg.payload.decode()

    message_dict = json.loads(message)

    mac_address = message_dict["mac_address"]
    timestamp = message_dict["timestamp"]
    temperature = message_dict["temperature"]
    humidity = message_dict["humidity"]

    # Create new time series for temperature and humidity, if they don't exist
    try:
        redis_client.ts.create(str(mac_address)+':temperature')
    except:
        pass
    try:
        redis_client.ts.create(str(mac_address)+':humidity')
    except:
        pass

    redis_client.ts().add(str(mac_address)+':temperature', timestamp, temperature)
    redis_client.ts().add(str(mac_address)+':humidity', timestamp, humidity)

    # Print the message and its topic
    print(f"Received message '{message}' on topic {msg.topic}")

## Create a MQTT client and set the callbacks to it

In [5]:
client = mqtt.Client()

client.on_connect = on_connect
client.on_message = on_message

## Connect to the MQTT broker and wait for messages

In [6]:
client.connect('mqtt.eclipseprojects.io', 1883)

# Start the client loop to process network events and call the callbacks
client.loop_forever()

Connected with result code 0
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815119315, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815121569, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815123824, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815126078, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815128332, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815132848, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45f01e89914", "timestamp": 1737815135102, "temperature": 20, "humidity": 57}' on topic s345139
Received message '{"mac_address": "0xe45

KeyboardInterrupt: 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b4ef5aa4-3f71-4837-91f1-c6fd9810a7ea' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>